# PoliMillionaire Game Tests

Run live games with selected local models, one after another.

No generated-answer API is used.


## Setup

Find the repo and local model cache.


In [1]:
import gc
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "polimillionaire" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the project folder.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
for path in [REPO_ROOT / "src", REPO_ROOT]:
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

os.environ.setdefault("HF_HOME", str(REPO_ROOT / "data" / "hf_home"))
os.environ.setdefault(
    "HUGGINGFACE_HUB_CACHE", str(REPO_ROOT / "data" / "hf_home" / "hub")
)

print("Repo:", REPO_ROOT)
print("HF cache:", os.environ["HF_HOME"])


Repo: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire
HF cache: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/data/hf_home


## Settings

Pick models here. Each selected model gets its own game run.


In [2]:
RUN_API_CHECK = True
RUN_LIVE_GAME = True
PROMPT_FOR_CREDENTIALS = False

API_URL = "http://131.175.15.22:51111/"
COMPETITION_ID = 0

MODELS_TO_RUN = [
    {"label": "Gemma 4 E2B", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma 4 E4B", "model_id": "google/gemma-4-E4B-it", "run": False},
    {"label": "Qwen 2.5 3B", "model_id": "Qwen/Qwen2.5-3B-Instruct", "run": False},
    {"label": "Qwen 2.5 7B", "model_id": "Qwen/Qwen2.5-7B-Instruct", "run": True},
]

SAFE_DELAY_SECONDS = 2.0
ANSWER_TIMEOUT_SECONDS = 25.0

print("API check:", RUN_API_CHECK)
print("Live game:", RUN_LIVE_GAME)
print("Competition:", COMPETITION_ID)
print("Models:")
for item in MODELS_TO_RUN:
    print("-", item["label"], item["model_id"], "run=", item["run"])


API check: True
Live game: True
Competition: 0
Models:
- Gemma 4 E2B google/gemma-4-E2B-it run= False
- Gemma 4 E4B google/gemma-4-E4B-it run= False
- Qwen 2.5 3B Qwen/Qwen2.5-3B-Instruct run= False
- Qwen 2.5 7B Qwen/Qwen2.5-7B-Instruct run= True


## Login

Use environment variables, Colab secrets, or prompt mode.


In [3]:
import getpass

from dotenv import load_dotenv

from millionaire_client import AuthenticationError, MillionaireClient

# Load environment variables from .env file
load_dotenv()

USERNAME = os.environ.get("POLIMILLIONAIRE_USERNAME")
PASSWORD = os.environ.get("POLIMILLIONAIRE_PASSWORD")

try:
    from google.colab import userdata

    USERNAME = USERNAME or userdata.get("POLIMILLIONAIRE_USERNAME")
    PASSWORD = PASSWORD or userdata.get("POLIMILLIONAIRE_PASSWORD")
except Exception:
    pass

if PROMPT_FOR_CREDENTIALS and not USERNAME:
    USERNAME = input("PoliMillionaire username: ").strip()
if PROMPT_FOR_CREDENTIALS and not PASSWORD:
    PASSWORD = getpass.getpass("PoliMillionaire password: ")

print("Username configured:", bool(USERNAME))
print("Password configured:", bool(PASSWORD))


Username configured: True
Password configured: True


## Competitions

Turn on `RUN_API_CHECK` to list games.


In [4]:
client = None

if RUN_API_CHECK:
    if not USERNAME or not PASSWORD:
        raise RuntimeError("Set credentials first.")
    try:
        client = MillionaireClient(API_URL)
        user = client.login(USERNAME, PASSWORD)
        print("Logged in as", user.username)
        for competition in client.competitions.list_all():
            print(competition.id, competition.name, competition.max_levels)
    except AuthenticationError as exc:
        client = None
        print("Login failed:", exc)
else:
    print("API check skipped.")


Logged in as promptPotamo
0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15
4 Philosophy and Psychology 15
5 News 15


## Run Selected Models

Each model warms up, plays if enabled, then unloads.


In [5]:
# !pip install  -qU colab-xterm
# %load_ext colabxterm
# # It may ask you to restart, accept and run this cell again

### For Colab: Run the following commands on the terminal below:
To spawn the server deamon please run the cell with ```%xterm```. It will create a concurrent shell where we can paste the following commands but, at the same time, also run other cells!

```sudo apt-get install zstd```

```curl -fsSL https://ollama.com/install.sh | sh```

```ollama serve & ollama pull llama3```

In [6]:
# %xterm

In [8]:
# ============================================================
# PHASE 1 REFACTORED TRIVIA AGENT
# ============================================================

# pip install -q \
#   langchain \
#   langchain-core \
#   langchain-community \
#   langchain-ollama \
#   ddgs \
#   numexpr \
#   pydantic \
#   rank_bm25 \
#   beautifulsoup4 \
#   requests

# ============================================================
# IMPORTS
# ============================================================

import logging
import math
import re
from dataclasses import dataclass, field
from enum import Enum
from typing import Any

import numexpr as ne
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("trivia-agent")

# ============================================================
# DATA STRUCTURES
# ============================================================


@dataclass(frozen=True)
class AnswerOption:
    id: int
    text: str


@dataclass(frozen=True)
class Question:
    id: int
    text: str
    options: list[AnswerOption]
    level: int = 0
    metadata: dict[str, Any] = field(default_factory=dict)


# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "qwen2.5:7b-instruct-q4_K_M"

TEMPERATURE = 0

MAX_DDG_RESULTS = 8
BM25_TOP_K = 4

TIMEOUT_DDG = 10

MEMORY_CONFIDENCE_THRESHOLD = 0.90

# ============================================================
# LLM
# ============================================================

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
)

# ============================================================
# OUTPUT SCHEMAS
# ============================================================


class TriviaAnswer(BaseModel):
    answer: int = Field(description="Option ID")
    confidence: float = Field(ge=0.00, le=1.00, description="Confidence score decimal numberbetween 0.00 and 1.00, 2 decimals precision")
    evidence: str = Field(description="Short evidence-based justification")

class MemoryAttempt(BaseModel):
    answer: int
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str


class VerificationResult(BaseModel):
    supported: bool
    explanation: str


class Route(str, Enum):
    CALCULATION = "calculation"
    RETRIEVAL = "retrieval"


class RouteDecision(BaseModel):
    route: Route


# ============================================================
# STRUCTURED LLMs
# ============================================================

memory_llm = llm.with_structured_output(MemoryAttempt)

reasoning_llm = llm.with_structured_output(TriviaAnswer)

verification_llm = llm.with_structured_output(VerificationResult)

router_llm = llm.with_structured_output(RouteDecision)

# ============================================================
# HELPERS
# ============================================================

RECENT_PATTERNS = [
    r"2024",
    r"2025",
    r"2026",
    r"current",
    r"latest",
    r"recent",
    r"today",
    r"now",
    r"winner",
    r"champion",
    r"president",
]

MATH_PATTERNS = [
    r"\d+\s*[\+\-\*/]\s*\d+",
    r"\d+\s*percent",
    r"sqrt|sin|cos|tan|log",
    r"interest|compound",
    r"calculate|compute|evaluate",
]

def is_math_question(text: str) -> bool:
    t = text.lower()
    return any(re.search(p, t) for p in MATH_PATTERNS)

def is_recent_question(text: str) -> bool:
    text = text.lower()
    return any(re.search(p, text) for p in RECENT_PATTERNS)


def build_mcq(question: Question) -> str:
    options = "\n".join(f"{option.id}) {option.text}" for option in question.options)
    mcq_text = f"{question.text}\n{options}"
    return mcq_text


# ============================================================
# ROUTER
# ============================================================


def route_question(question: str) -> Route:
    prompt = f"""
Classify the question into ONE route.

1. CALCULATION
Use ONLY if numerical computation is required.

2. RETRIEVAL
Use for factual, conceptual, historical,
scientific, or explanatory questions.

Question:
{question}
"""

    result = router_llm.invoke(prompt)

    return result.route


# ============================================================
# MEMORY ATTEMPT
# ============================================================

MEMORY_PROMPT = """
You are a careful multiple-choice trivia solver.

Use ONLY your internal knowledge.

Confidence rules:
- 0.95-1.00:
  Explicitly known fact.

- 0.70-0.89:
  Strong memory but some uncertainty.

- 0.40-0.69:
  Weak recollection.

- 0.00-0.39:
  Mostly guessing.

If uncertain:
- lower confidence significantly.
"""

MEMORY_REASONING_PROMPT = """
QUESTION:
{question}

Return the most likely option.
"""

memory_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", MEMORY_PROMPT),
        ("human", MEMORY_REASONING_PROMPT),
    ]
)

memory_chain = memory_prompt | memory_llm


def memory_attempt(question: str) -> MemoryAttempt:
    return memory_chain.invoke({"question": question})


# ============================================================
# WEB SEARCH
# ============================================================


def ddg_search(query: str) -> list[dict]:
    with DDGS(timeout=TIMEOUT_DDG) as ddgs:
        results = list(
            ddgs.text(
                query,
                max_results=MAX_DDG_RESULTS,
            )
        )

    return results


def dedupe_results(results: list[dict]) -> list[dict]:
    seen = set()
    deduped = []

    for r in results:
        url = (r.get("href") or "").strip()

        if not url:
            continue

        norm = url.lower().rstrip("/")

        if norm in seen:
            continue

        seen.add(norm)

        deduped.append(r)

    return deduped


def results_to_documents(results: list[dict]) -> list[Document]:
    docs = []

    for r in results:
        title = r.get("title", "")
        snippet = r.get("body", "")
        url = r.get("href", "")

        text = f"{title}\n{snippet}"

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "title": title,
                    "snippet": snippet,
                    "url": url,
                },
            )
        )

    return docs


def rerank(query: str, docs: list[Document]) -> list[Document]:
    retriever = BM25Retriever.from_documents(
        docs,
        k=BM25_TOP_K,
    )

    return retriever.invoke(query)


def format_evidence(docs: list[Document]) -> str:
    blocks = []

    for i, d in enumerate(docs, 1):
        blocks.append(
            f"[{i}] TITLE: {d.metadata['title']}\nSNIPPET: {d.metadata['snippet']}"
        )

    return "\n\n".join(blocks)


def retrieve(question: Question) -> str:
    query = build_mcq(question)

    log.info(f"SEARCH QUERY: {query}")

    raw_results = ddg_search(query)

    raw_results = dedupe_results(raw_results)

    docs = results_to_documents(raw_results)

    reranked = rerank(query, docs)

    evidence = format_evidence(reranked)

    return evidence[:2500]


# ============================================================
# MATH
# ============================================================

math_parser_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Extract ONLY a valid numexpr expression. Use python numexpr syntax."
     "Return ONLY the expression, no text."),
    ("human", "{question}")
])

math_parser_chain = math_parser_prompt | llm

def extract_expression(question: str) -> str:
    expr = math_parser_chain.invoke({"question": question}).content.strip()
    return expr


def run_math(question: str) -> str:
    expr = extract_expression(question)

    log.info(f"[MATH] EXPRESSION: {expr}")

    try:
        result = ne.evaluate(
            expr,
            local_dict={
                "pi": math.pi,
                "e": math.e,
            },
        )

        result = result.item()

        return f"Expression: {expr}\nResult: {result}"

    except Exception as e:
        return f"Math error: {e}"


# ============================================================
# REASONING
# ============================================================

SYSTEM_PROMPT = """
You are a careful multiple-choice trivia solver.

Rules:
1. Use evidence primarily.
2. Use internal knowledge only to supplement evidence.
3. Compare ALL options carefully.
4. Select EXACTLY one option ID.
5. Lower confidence if evidence is weak.

"""

REASONING_PROMPT = """
QUESTION:
{question}

EVIDENCE:
{evidence}

Confidence score is a decimal number between 0.00 (lowest) and 1.00 (highest).
Return the best option.
"""

reasoning_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("human", REASONING_PROMPT),
    ]
)

reasoning_chain = reasoning_prompt | reasoning_llm


def reason(question: str, evidence: str) -> TriviaAnswer:
    return reasoning_chain.invoke(
        {
            "question": question,
            "evidence": evidence,
        }
    )


# ============================================================
# VERIFICATION
# ============================================================

VERIFY_PROMPT = """
QUESTION:
{question}

EVIDENCE:
{evidence}

PROPOSED ANSWER:
{answer}

Does the evidence explicitly support the answer?
Be strict.
"""

verification_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a strict fact verifier."),
        ("human", VERIFY_PROMPT),
    ]
)

verification_chain = verification_prompt | verification_llm


def verify_answer(
    question: str,
    evidence: str,
    answer: TriviaAnswer,
) -> VerificationResult:
    return verification_chain.invoke(
        {
            "question": question,
            "evidence": evidence,
            "answer": answer.model_dump_json(indent=2),
        }
    )


# ============================================================
# MAIN PIPELINE
# ============================================================


def ask_question(q: Question):
    log.info("=" * 80)
    log.info("QUESTION")

    mcq = build_mcq(q)

    log.info(mcq)

    # ========================================================
    # MEMORY ATTEMPT
    # ========================================================

    memory_result = memory_attempt(mcq)

    log.info("MEMORY RESULT")
    log.info(memory_result)

    needs_retrieval = (
        memory_result.confidence < MEMORY_CONFIDENCE_THRESHOLD
        or is_recent_question(mcq)
    )

    if not needs_retrieval:
        log.info("HIGH CONFIDENCE MEMORY ANSWER")

        return memory_result

    # ========================================================
    # ROUTING
    # ========================================================

    route = route_question(mcq)

    log.info(f"ROUTE: {route}")

    # ========================================================
    # CALCULATION
    # ========================================================

    if route == Route.CALCULATION:
        evidence = run_math(q.text)

    # ========================================================
    # RETRIEVAL
    # ========================================================

    else:
        evidence = retrieve(q)

    log.info("EVIDENCE")
    log.info(evidence[:1500])

    # ========================================================
    # FINAL REASONING
    # ========================================================

    result = reason(mcq, evidence)

    log.info("FINAL RESULT")
    log.info(result)

    # ========================================================
    # VERIFICATION
    # ========================================================

    verification = verify_answer(
        mcq,
        evidence,
        result,
    )

    log.info("VERIFICATION")
    log.info(verification)

    if not verification.supported:
        result.confidence *= 0.5

    return result


# ============================================================
# EXAMPLE
# ============================================================

factul_question = Question(
    id=1,
    text="Which country won the FIFA World Cup in 2018?",
    options=[
        AnswerOption(0, "Germany"),
        AnswerOption(1, "Argentina"),
        AnswerOption(2, "France"),
        AnswerOption(3, "Brazil"),
    ],
)

# RECENT QUESTION
recent_question = Question(
    id=3,
    text="Who won the UEFA Champions League in 2025?",
    options=[
        AnswerOption(0, "Real Madrid"),
        AnswerOption(1, "Manchester City"),
        AnswerOption(2, "PSG"),
        AnswerOption(3, "Bayern Munich"),
    ],
)

calculation_question = Question(
    id=5,
    text="What is the result of cosine of 90 degrees?",
    options=[
        AnswerOption(0, "45"),
        AnswerOption(1, "pi/2"),
        AnswerOption(2, "0"),
        AnswerOption(3, "1"),
    ],
)

calculation_question_2 = Question(
    id=4,
    text="At an annual interest rate of 95%, what would 50 dollars be worth in 10 years?",
    options=[
        AnswerOption(0, "39748"),
        AnswerOption(1, "525"),
        AnswerOption(2, "3475"),
        AnswerOption(3, "500"),
    ],
)

question_5 = Question(
    id=10,
    text="The following best describes Clark Gable's role in World War II?",
    options=[
        AnswerOption(0, "He was a military general"),
        AnswerOption(1, "He remained in Hollywood and did not participate"),
        AnswerOption(2, "He was a spy for the United States"),
        AnswerOption(3, "He served in the United States Army Air Forces"),
    ],
)


result = ask_question(recent_question)

print("\n")
print(result.model_dump_json(indent=2))

INFO:trivia-agent:================================================================================
INFO:trivia-agent:QUESTION
INFO:trivia-agent:Who won the UEFA Champions League in 2025?
0) Real Madrid
1) Manchester City
2) PSG
3) Bayern Munich
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:MEMORY RESULT
INFO:trivia-agent:answer=0 confidence=0.4 reasoning='Real Madrid has a strong history in the UEFA Champions League and is known for their consistent performance, but predicting future events with certainty is challenging.'
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:ROUTE: Route.RETRIEVAL
INFO:trivia-agent:SEARCH QUERY: Who won the UEFA Champions League in 2025?
0) Real Madrid
1) Manchester City
2) PSG
3) Bayern Munich
INFO:primp:response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=Who%20won%20the%20UEFA%20Champions%20League%20in%202025%3F%0A0%



{
  "answer": 2,
  "confidence": 0.425,
  "evidence": "[2] [3]"
}


In [ ]:
from abc import ABC, abstractmethod
from typing import Any, Protocol

from src.polimillionaire.strategies import parse_answer_prediction


class BaseStrategy(ABC):
    name = "base"

    @abstractmethod
    def answer(self, question: Question) -> AnswerPrediction:
        raise NotImplementedError


class LocalLLM(Protocol):
    model_name: str

    def generate(self, prompt: str, **kwargs: object) -> str: ...


class OllamaStrategy(BaseStrategy):
    name = "Ollama-Gwen2.5-7b"

    def __init__(self):
        self.llm = ChatOllama(model="gwen2.5:7b", temperature=0)

    def answer(self, question: Question) -> AnswerPrediction:
        prediction = ask_question(question)

        # raw_text = self.llm.generate(build_prompt(question))
        prediction = parse_answer_prediction(
            str(prediction), question, strategy_name=self.name
        )
        prediction.metadata["strategy"] = self.name
        prediction.metadata["model_name"] = getattr(self.llm, "model_name", "unknown")
        prediction.metadata["device"] = getattr(self.llm, "device_summary", "unknown")
        return prediction

In [ ]:
from datetime import datetime, timezone

from polimillionaire.runner import GameRunner, RunLogger
from polimillionaire.strategies import GemmaLLMConfig, GemmaStrategy
from polimillionaire.types import AnswerOption, Question

warmup_question = Question(
    id=1,
    text="What is 2 + 2?",
    options=[
        AnswerOption(0, "3"),
        AnswerOption(1, "4"),
        AnswerOption(2, "5"),
        AnswerOption(3, "22"),
    ],
)


def clear_model_memory():
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        if (
            hasattr(torch, "mps")
            and hasattr(torch.backends, "mps")
            and torch.backends.mps.is_available()
        ):
            torch.mps.empty_cache()
    except Exception as exc:
        print("Cleanup warning:", exc)


def make_strategy(model_id: str) -> GemmaStrategy:
    config = GemmaLLMConfig(
        model_id=model_id,
        inference_backend="auto_model",
        device_map="auto",
        dtype="auto",
        max_new_tokens=8,
        temperature=0.0,
        do_sample=False,
        num_beams=1,
        seed=42,
        generation_max_time_seconds=18.0,
        timeout_seconds=120.0,
    )
    return GemmaStrategy(model_config=config)


def clean_name(label: str) -> str:
    return "_".join(label.lower().split())


if RUN_LIVE_GAME and (not USERNAME or not PASSWORD):
    raise RuntimeError("Set credentials first.")
if RUN_LIVE_GAME and client is None:
    client = MillionaireClient(API_URL)
    client.login(USERNAME, PASSWORD)

run_results = []
for item in MODELS_TO_RUN:
    if not item.get("run", False):
        print("Skipped", item["label"])
        continue

    strategy = None
    try:
        print()
        print("Model:", item["label"])
        # strategy = make_strategy(item["model_id"])
        strategy = OllamaStrategy()

        warmup = strategy.answer(warmup_question)

        print("warmup option:", warmup.option_id, warmup.answer_text)
        print("fallback:", warmup.metadata.get("fallback"))
        print("device:", warmup.metadata.get("device"))
        if warmup.metadata.get("fallback"):
            raise RuntimeError(f"Warm-up failed for {item['label']}.")

        result = {
            "label": item["label"],
            "model_id": item["model_id"],
            "warmup_option": warmup.option_id,
        }

        if RUN_LIVE_GAME:
            run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
            log_path = (
                REPO_ROOT
                / "results"
                / "runs"
                / f"{clean_name(item['label'])}_{run_id}.jsonl"
            )
            runner = GameRunner(
                client=client,
                safe_delay_seconds=SAFE_DELAY_SECONDS,
                answer_timeout_seconds=ANSWER_TIMEOUT_SECONDS,
                logger=RunLogger(log_path),
            )
            game = runner.play(COMPETITION_ID, strategy)
            result.update(
                {
                    "current_level": game.current_level,
                    "earned_amount": game.earned_amount,
                    "log_path": str(log_path),
                }
            )
            print("Reached level:", game.current_level)
            print("Earned amount:", game.earned_amount)
            print("Log path:", log_path)
        else:
            print("Live game skipped for", item["label"])

        run_results.append(result)
    finally:
        del strategy
        clear_model_memory()
        print("Cleared model memory after", item["label"])

run_results


INFO:trivia-agent:=== QUESTION ===
INFO:trivia-agent:What is 2 + 2?
0. 3
1. 4
2. 5
3. 22


Skipped Gemma 4 E2B
Skipped Gemma 4 E4B
Skipped Qwen 2.5 3B

Model: Qwen 2.5 7B


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:=== DIRECT ANSWER ATTEMPT ===
INFO:trivia-agent:answer=1 confidence=0.95 reasoning='The question asks for the sum of 2 + 2. The correct answer is 4. Option 1 (4) is clearly the right choice among the given options.'
INFO:trivia-agent:=== QUESTION ===
INFO:trivia-agent:How does the film 'The Shawshank Redemption' primarily convey its message of hope?
0. By Andy's escape from the prison
1. By Red's final speech
2. Through the prison's harsh environment
3. Through explicit dialogue


warmup option: 1 4
fallback: False
device: unknown


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:=== DIRECT ANSWER ATTEMPT ===
INFO:trivia-agent:answer=1 confidence=0.65 reasoning="Red's final speech conveys the message of hope through his reflections and the release letter Andy mailed to him, symbolizing freedom and hope even in the most oppressive environments. While Andy's escape is a pivotal moment, it might be seen as more of an action than a primary method for conveying the film's overarching theme of hope. The prison environment and explicit dialogue are also elements but not as explicitly focused on hope as Red’s speech."
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:ROUTE: Route.RETRIEVAL
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:DDG query: how does the film The Shawshank Redemption primarily convey its message of hope through scenes or themes
INFO:primp:response: https

Reached level: 3
Earned amount: 200
Log path: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/results/runs/qwen_2.5_7b_20260526_202542.jsonl
Cleared model memory after Qwen 2.5 7B


[{'label': 'Qwen 2.5 7B',
  'model_id': 'Qwen/Qwen2.5-7B-Instruct',
  'warmup_option': 1,
  'current_level': 3,
  'earned_amount': 200,
  'log_path': '/Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/results/runs/qwen_2.5_7b_20260526_202542.jsonl'}]

INFO:trivia-agent:answer=3 confidence=0.9 evidence="The evidence from multiple sources, including Wikipedia and Hertel Walls, clearly states that after Humphrey Bogart's death in 1957, Frank Sinatra became the leader of the Rat Pack. This information is explicitly supported by the snippets provided."
INFO:trivia-agent:=== RESULT (STRUCTURED) ===
INFO:trivia-agent:{
  "answer": 3,
  "confidence": 0.9,
  "evidence": "The evidence from multiple sources, including Wikipedia and Hertel Walls, clearly states that after Humphrey Bogart's death in 1957, Frank Sinatra became the leader of the Rat Pack. This information is explicitly supported by the snippets provided."
}


## Results

Read recent game logs.


In [ ]:
from polimillionaire.runner import load_jsonl, summarize_attempts

run_logs = sorted(
    (REPO_ROOT / "results" / "runs").glob("*.jsonl"),
    key=lambda path: path.stat().st_mtime,
)

for path in run_logs[-5:]:
    print(path.name)

if run_logs:
    rows = load_jsonl(run_logs[-1])
    print("latest:", run_logs[-1].name)
    print(summarize_attempts(rows))
else:
    print("No logs found.")


## Done

Use `run_results` and the JSONL logs for comparison.
